# 复现Alpaca模型
下面从0到1复现Alpaca模型的训练过程。

## 环境准备
- GPUs: 8 卡 A800 80GB GPUs
- Python: 3.10 (需要先升级OpenSSL到1.1.1t版本（点击下载OpenSSL），然后再编译安装Python)Python
- NVIDIA驱动程序版本: 470.161.03，根据不同型号选择不同的驱动程序。
- CUDA工具包: 11.3
- NCCL: 2.9.9-1
- cuDNN: v8.2.0

创建一个虚拟环境，并激活，然后安装`PyTorch`, whl下载：[https://download.pytorch.org/whl/torch_stable.html](https://download.pytorch.org/whl/torch_stable.html)
```bash 
pip install torch-1.12.1+cu113-cp310-cp310-linux_x86_64.whl
pip install torchvision-0.13.1+cu113-cp310-cp310-linux_x86_64.whl
```

安装transformers，目前，LLaMA相关的实现并没有发布对应的版本，但是已经合并到主分支了，因此，我们需要切换到对应的commit，从源代码进行相应的安装。

```bash
cd transformers
git checkout 0041be5 
pip install .
```

安装apex。
```bash
git clone https://github.com/NVIDIA/apex.git
cd apex
git checkout 22.04-dev
pip install -v --disable-pip-version-check --no-cache-dir --global-option="--cpp_ext" --global-option="--cuda_ext" ./
```

安装stanford_alpaca依赖库。\
首先克隆stanford_alpaca代码仓库，然后安装requirements：
```bash
git clone https://github.com/tatsu-lab/stanford_alpaca.git
pip install -r requirements.txt
```

## 模型格式转换
将LLaMA原始权重文件转换为Transformers库对应的模型文件格式。进入到`transformers`目录中：
```bash
python src/transformers/models/llama/convert_llama_weights_to_hf.py \ 
--input_dir /data/nfs/guodong.li/pretrain/llama-model \
--model_size 7B \
--output_dir /data/nfs/guodong.li/pretrain/hf-llama-model
```
转换之后会生成tokenizer和llama-7b（模型权重文件）两个目录。
LLaMA 分词器（tokenizer）基于 sentencepiece分词工具。 sentencepiece在解码序列时，如果第一个token是单词（例如：Banana）开头，则tokenizer不会在字符串前添加前缀空格。 要让tokenizer输出前缀空格，请在LlamaTokenizer对象或tokenizer配置中设置decode_with_prefix_space=True。

## 数据准备
Stanford Alpaca中的alpaca_data.json文件即是他们用于训练的指令数据集，我们可以直接使用该数据集进行模型精调。

## 模型微调
微调模型的超参数如下：
| 超参数 |	LLaMA-7B|
| -- | -- |
|Batch size	128	|
|学习率|	2e-5|
|Epochs|	3|
|Max length|	512	|
|Weight decay|	0|

In [ ]:
torchrun --nproc_per_node=8 --master_port=25001 train.py \
    --model_name_or_path  /data/nfs/guodong.li/pretrain/hf-llama-model/llama-7b \
    --data_path /data/nfs/guodong.li/data/alpaca_data_cleaned.json \
    --bf16 True \
    --output_dir /data/nfs/guodong.li/output/alpaca/sft_7b \
    --num_train_epochs 1 \
    --per_device_train_batch_size 4 \
    --per_device_eval_batch_size 4 \
    --gradient_accumulation_steps 8 \
    --evaluation_strategy "no" \
    --save_strategy "steps" \
    --save_steps 2000 \
    --save_total_limit 1 \
    --learning_rate 2e-5 \
    --weight_decay 0. \
    --warmup_ratio 0.03 \
    --lr_scheduler_type "cosine" \
    --logging_steps 1 \
    --report_to "tensorboard" \
    --fsdp "full_shard auto_wrap" \
    --fsdp_transformer_layer_cls_to_wrap 'LlamaDecoderLayer' \
    --tf32 True